In [ ]:
# 키워드 기반 사용자 질의 1차 분류 함수

def classify_intent_by_keywords(user_query, category_schema):
    user_query = str(user_query)

    matched_results = []

    for category_info in category_schema:
        category = category_info["category"]
        keywords = category_info["all_keywords"]

        matched_keywords = [
            keyword for keyword in keywords
            if keyword and keyword in user_query
        ]

        if matched_keywords:
            matched_results.append({
                "category": category,
                "matched_keywords": matched_keywords,
                "match_count": len(matched_keywords),
                "mention_ratio": category_info["mention_ratio"]
            })

    if not matched_results:
        return None

    matched_results = sorted(
        matched_results,
        key=lambda x: (x["match_count"], x["mention_ratio"]),
        reverse=True
    )

    return matched_results[0]


# 테스트

test_queries = [
    "여자친구랑 데이트하기 좋은 감성 카페 코스 추천해줘",
    "부모님 모시고 조용히 힐링할 수 있는 곳 추천해줘",
    "친구들이랑 캠핑이나 레포츠 할 만한 곳 알려줘",
    "아이와 함께 체험할 수 있는 관광지 추천해줘"
]

for query in test_queries:
    travel_result = classify_intent_by_keywords(
        query,
        travel_category_schema
    )

    companion_result = classify_intent_by_keywords(
        query,
        companion_category_schema
    )

    print("질의:", query)
    print("여행유형:", travel_result)
    print("동반유형:", companion_result)
    print("-" * 80)

In [ ]:
# LLM 프롬프트 템플릿 예시

def make_intent_extraction_prompt(user_query, llm_intent_category_schema):
    prompt = f"""
너는 남양주시 관광 추천 시스템의 사용자 의도 분석기이다.

사용자 질의에서 여행유형과 동반유형을 추출해야 한다.

반드시 아래 카테고리 중에서만 선택한다.
판단이 어려우면 null을 반환한다.

[여행유형 카테고리]
{json.dumps(llm_intent_category_schema["travel_type_categories"], ensure_ascii=False, indent=2)}

[동반유형 카테고리]
{json.dumps(llm_intent_category_schema["companion_type_categories"], ensure_ascii=False, indent=2)}

[출력 형식]
{{
  "travel_type": "카테고리명 또는 null",
  "companion_type": "카테고리명 또는 null",
  "travel_keywords": ["질의에서 추출한 여행 관련 키워드"],
  "companion_keywords": ["질의에서 추출한 동반 관련 키워드"],
  "confidence": 0.0
}}

[사용자 질의]
{user_query}
"""
    return prompt


sample_prompt = make_intent_extraction_prompt(
    "여자친구랑 데이트하기 좋은 감성 카페 코스 추천해줘",
    llm_intent_category_schema
)

print(sample_prompt[:3000])